# LeetCode 462 — Minimum Moves to Equal Array Elements II bằng Python

Link đề: https://leetcode.com/problems/minimum-moves-to-equal-array-elements-ii/ · Độ khó: **Medium** · Chủ đề: Array, Math, Sorting

Skeleton LeetCode:

```python
class Solution:
    def minMoves2(self, nums: List[int]) -> int:
        ...
```

Bài này dạy một sự thật rất đẹp: **tổng khoảng cách tới một điểm trên trục số nhỏ nhất khi điểm đó là trung vị (median)**. Từ đó một đề tưởng khó trở thành "sắp xếp rồi lấy phần tử giữa". Notebook đi từ thử mọi đích, đến trung vị, ghép cặp hai đầu, `quickselect` `O(n)` và tìm kiếm nhị phân trên độ dốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi lượt **tăng hoặc giảm một phần tử đi 1**; cần số lượt ít nhất để mọi phần tử **bằng nhau**.
2. Nắm các công cụ Python: `abs`, `sum` với biểu thức sinh, `sorted`/`list.sort`, chỉ số phần tử giữa `n // 2`, hai con trỏ từ hai đầu, `zip` với `reversed`, phân hoạch (partition) kiểu `quickselect`, và tìm kiếm nhị phân trên **giá trị**.
3. Đi từ **thử mọi giá trị đích `O(n·R)`** → **sắp xếp + trung vị `O(n log n)`** → **ghép cặp hai đầu** → **quickselect `O(n)` trung bình** → **nhị phân trên độ dốc `O(n log R)`**.
4. Hiểu mấu chốt: tổng khoảng cách `f(x) = Σ|a_i - x|` là hàm **lồi**, đạt cực tiểu ở **trung vị**; cặp phần tử đầu-cuối luôn tốn ít nhất `a_max - a_min` bất kể đích nằm đâu giữa chúng.
5. Tránh các lỗi kinh điển: dùng **trung bình** thay vì trung vị, lấy phần tử giữa của mảng **chưa sắp xếp**, nhầm với bài 453 (đích là `min`), chọn sai chỉ số giữa khi `n` lẻ, và vét cạn quá chậm.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` kích thước `n`. Trả về **số lượt ít nhất** để làm cho mọi phần tử bằng nhau. Mỗi lượt bạn được **tăng hoặc giảm một phần tử của mảng đi `1`**.

Các test được thiết kế sao cho đáp án vừa **số nguyên 32-bit**.

**Ví dụ 1**

- Input: `nums = [1,2,3]`
- Output: `2`
- Giải thích: `[1,2,3] → [2,2,3] → [2,2,2]` (tăng `1` lên `2`, giảm `3` xuống `2`).

**Ví dụ 2**

- Input: `nums = [1,10,2,9]`
- Output: `16`

**Ràng buộc:**

- `n == nums.length`
- `1 <= nums.length <= 10^5`
- `-10^9 <= nums[i] <= 10^9`

In [1]:
from typing import List
import random
import time
from collections import deque
from itertools import product

examples = [([1, 2, 3], 2), ([1, 10, 2, 9], 16)]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[1, 2, 3] -> kỳ vọng 2
[1, 10, 2, 9] -> kỳ vọng 16


## 3. Kiến thức nền

### 3.1. Khoảng cách trên trục số và `abs`

Khoảng cách giữa hai số `a` và `x` trên trục số là `|a - x|`, trong Python `abs(a - x)`. Đưa một phần tử `a` về giá trị `x` bằng các lượt `±1` tốn **đúng** `|a - x|` lượt. Vậy tổng số lượt để đưa **cả mảng** về `x` là `Σ|a_i - x|`.

In [2]:
a = [1, 10, 2, 9]
for x in (1, 2, 5, 9, 10):
    print(f"đích x={x}: tổng lượt =", sum(abs(v - x) for v in a))

đích x=1: tổng lượt = 18
đích x=2: tổng lượt = 16
đích x=5: tổng lượt = 16
đích x=9: tổng lượt = 16
đích x=10: tổng lượt = 18


### 3.2. `sorted` và `list.sort`

`sorted(a)` trả về list mới đã sắp xếp (không đổi `a`); `a.sort()` sắp xếp **tại chỗ** và trả `None`. Sắp xếp tốn `O(n log n)`. Sau khi sắp xếp, `a[0]` là nhỏ nhất, `a[-1]` là lớn nhất.

In [3]:
a = [3, 1, 2]
b = sorted(a)
print(a, b)                  # [3, 1, 2] [1, 2, 3]
r = a.sort()
print(a, r)                  # [1, 2, 3] None

[3, 1, 2] [1, 2, 3]
[1, 2, 3] None


### 3.3. Trung vị (median) và chỉ số `n // 2`

Trung vị là phần tử **ở giữa** sau khi sắp xếp. Với `n` lẻ chỉ có một phần tử giữa, chỉ số `n // 2`. Với `n` chẵn có hai phần tử giữa, chỉ số `n // 2 - 1` và `n // 2`; **chọn một trong hai** đều cho cùng tổng khoảng cách (ta sẽ chứng minh). `//` là chia lấy phần nguyên.

In [4]:
for a in ([5, 1, 3], [4, 1, 3, 2]):
    s = sorted(a)
    n = len(s)
    print(s, "n =", n, "n//2 =", n // 2, "-> s[n//2] =", s[n // 2])

[1, 3, 5] n = 3 n//2 = 1 -> s[n//2] = 3
[1, 2, 3, 4] n = 4 n//2 = 2 -> s[n//2] = 3


### 3.4. Hai con trỏ từ hai đầu và `zip` với `reversed`

Duyệt cặp "đầu - cuối" của mảng đã sắp xếp: `i` từ trái, `j` từ phải, cùng tiến vào giữa. Hoặc `zip(s, reversed(s))` ghép `s[0]` với `s[-1]`, `s[1]` với `s[-2]`, … và dừng khi hết phần tử.

In [5]:
s = [1, 2, 9, 10]
i, j = 0, len(s) - 1
while i < j:
    print(f"cặp ({s[i]}, {s[j]}) chênh {s[j] - s[i]}")
    i += 1
    j -= 1
print(list(zip(s, reversed(s))))     # [(1, 10), (2, 9), (9, 2), (10, 1)]  (mỗi cặp xuất hiện 2 lần)

cặp (1, 10) chênh 9
cặp (2, 9) chênh 7
[(1, 10), (2, 9), (9, 2), (10, 1)]


### 3.5. Hàm lồi và tìm kiếm nhị phân trên giá trị

`f(x) = Σ|a_i - x|` là tổng các hàm "chữ V", nên có dạng **lồi**: giảm dần rồi tăng dần. Độ dốc giữa `x` và `x+1` là `f(x+1) - f(x) = (số phần tử ≤ x) - (số phần tử > x)` và **không giảm** khi `x` tăng. Có thể nhị phân tìm điểm độ dốc đổi từ âm sang không âm.

In [6]:
a = [1, 10, 2, 9]
f = lambda x: sum(abs(v - x) for v in a)
for x in range(0, 12):
    slope = sum(1 for v in a if v <= x) - sum(1 for v in a if v > x)
    print(f"x={x:>2}  f={f(x):>3}  độ dốc f(x+1)-f(x) = {f(x + 1) - f(x):>2}  (đếm = {slope:>2})")

x= 0  f= 22  độ dốc f(x+1)-f(x) = -4  (đếm = -4)
x= 1  f= 18  độ dốc f(x+1)-f(x) = -2  (đếm = -2)
x= 2  f= 16  độ dốc f(x+1)-f(x) =  0  (đếm =  0)
x= 3  f= 16  độ dốc f(x+1)-f(x) =  0  (đếm =  0)
x= 4  f= 16  độ dốc f(x+1)-f(x) =  0  (đếm =  0)
x= 5  f= 16  độ dốc f(x+1)-f(x) =  0  (đếm =  0)
x= 6  f= 16  độ dốc f(x+1)-f(x) =  0  (đếm =  0)
x= 7  f= 16  độ dốc f(x+1)-f(x) =  0  (đếm =  0)
x= 8  f= 16  độ dốc f(x+1)-f(x) =  0  (đếm =  0)
x= 9  f= 16  độ dốc f(x+1)-f(x) =  2  (đếm =  2)
x=10  f= 18  độ dốc f(x+1)-f(x) =  4  (đếm =  4)
x=11  f= 22  độ dốc f(x+1)-f(x) =  4  (đếm =  4)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy nghĩ các phần tử là **người đứng trên một đường thẳng**, mọi người cần tập trung tại một điểm; mỗi bước đi `1` đơn vị.

- **Hai người** ở vị trí `a < b`: đứng ở bất kỳ điểm giữa `a` và `b` đều tốn `b - a`. Ra ngoài đoạn thì tốn **nhiều hơn**.
- **Ba người** `1, 2, 3`: nếu chọn `2` thì tốn `1 + 0 + 1 = 2`. Người đứng giữa không cần đi.
- **Bốn người** `1, 2, 9, 10`: ghép `(1, 10)` tốn tối thiểu `9` bất kể đích (miễn nằm giữa); ghép `(2, 9)` tốn tối thiểu `7`; tổng `16`. Chọn đích `2`...`9` đều đạt `16`.

Nhận xét then chốt: đích tốt nhất nằm **trong mọi đoạn** `[a_i, a_{n+1-i}]` (đoạn đầu-cuối lồng nhau); điểm chung của chúng chính là **trung vị**.

In [7]:
s = sorted([1, 10, 2, 9])
pairs = [(s[i], s[-1 - i]) for i in range(len(s) // 2)]
print(pairs, [b - a for a, b in pairs], sum(b - a for a, b in pairs))     # [(1,10),(2,9)] [9,7] 16

[(1, 10), (2, 9)] [9, 7] 16


## 5. Cách 1 — Brute Force / baseline: thử mọi giá trị đích

**Ý tưởng:** đích tối ưu chắc chắn nằm trong `[min, max]` (ra ngoài thì mọi phần tử cùng xa thêm). Thử **mọi số nguyên** `x` trong đoạn đó, tính `Σ|a_i - x|`, lấy nhỏ nhất.

In [8]:
def moves_try_all(nums):
    lo, hi = min(nums), max(nums)
    best = None
    for x in range(lo, hi + 1):
        cost = sum(abs(v - x) for v in nums)
        if best is None or cost < best:
            best = cost
    return best


for nums, expected in examples:
    print(nums, moves_try_all(nums), "kỳ vọng", expected)

[1, 2, 3] 2 kỳ vọng 2
[1, 10, 2, 9] 16 kỳ vọng 16


**Độ phức tạp:** `Time = O(n · R)` với `R = max - min + 1` (có thể `2·10^9`), `Space = O(1)`. Không khả thi với ràng buộc đề.

**Nút thắt:** ta thử cả những `x` rõ ràng vô ích (không thuộc mảng). Một lượt thử cũng tốn `O(n)` mà không tận dụng thông tin của lượt trước. Cần biết **đích tối ưu là gì** thay vì dò.

## 6. Ý tưởng thuật toán / bước đột phá

**Định lý.** Với mọi `x`, `Σ|a_i - x|` đạt cực tiểu khi `x` là **trung vị** của `a`.

**Chứng minh bằng ghép cặp.** Sắp xếp `a_1 ≤ a_2 ≤ … ≤ a_n`. Ghép `a_1` với `a_n`, `a_2` với `a_{n-1}`, … Với mỗi cặp `(a_i, a_j)`, `i < j`:

`|a_i - x| + |a_j - x| ≥ |a_j - a_i| = a_j - a_i`  (bất đẳng thức tam giác),

và **dấu bằng** xảy ra đúng khi `a_i ≤ x ≤ a_j`. Nếu `n` lẻ, phần tử giữa không ghép cặp, đóng góp `|a_m - x| ≥ 0`, bằng `0` khi `x = a_m`. Vậy

`Σ|a_i - x| ≥ Σ_{i < n/2} (a_{n+1-i} - a_i)`

và dấu bằng đạt được khi `x` thuộc **mọi** đoạn `[a_i, a_{n+1-i}]`, mà các đoạn này lồng nhau nên giao của chúng chính là đoạn giữa chứa trung vị. (Với `n` chẵn, mọi `x` giữa hai phần tử giữa đều tốt ⇒ chọn phần tử nào cũng được.)

**Hệ quả hai thuật toán:** (a) sắp xếp, lấy `x = a[n // 2]`, tổng `|a_i - x|`; (b) sắp xếp, **cộng các hiệu cặp** `Σ (a_{n-1-i} - a_i)`, không cần chọn `x`. Muốn tránh sắp xếp: tìm trung vị bằng `quickselect`, hoặc nhị phân trên giá trị.

In [9]:
a = sorted([1, 10, 2, 9, 4])
n = len(a)
m = a[n // 2]
print("trung vị:", m)
print("tổng |a - trung vị| =", sum(abs(v - m) for v in a))
print("tổng hiệu cặp đầu-cuối =", sum(a[n - 1 - i] - a[i] for i in range(n // 2)))
print("các đích khác:", {x: sum(abs(v - x) for v in a) for x in range(1, 11)})

trung vị: 4
tổng |a - trung vị| = 16
tổng hiệu cặp đầu-cuối = 16
các đích khác: {1: 21, 2: 18, 3: 17, 4: 16, 5: 17, 6: 18, 7: 19, 8: 20, 9: 21, 10: 24}


## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp + trung vị (cách chuẩn)

**Giải thích từng đoạn:**

1. Sắp xếp `nums`.
2. Lấy `k = nums[n // 2]` (trung vị).
3. Trả `Σ|v - k|`.

In [10]:
def moves_median(nums):
    a = sorted(nums)
    k = a[len(a) // 2]
    return sum(abs(v - k) for v in a)


for nums, expected in examples:
    print(nums, moves_median(nums), "kỳ vọng", expected)

[1, 2, 3] 2 kỳ vọng 2
[1, 10, 2, 9] 16 kỳ vọng 16


In [11]:
# Code nộp LeetCode — Cách 2
class Solution2:
    def minMoves2(self, nums: List[int]) -> int:
        nums.sort()
        k = nums[len(nums) >> 1]
        return sum(abs(v - k) for v in nums)


print(Solution2().minMoves2([1, 2, 3]), Solution2().minMoves2([1, 10, 2, 9]))

2 16


**Độ phức tạp Cách 2:** sắp xếp `O(n log n)`, cộng `O(n)` ⇒ `Time = O(n log n)`, `Space = O(1)` đến `O(n)` tuỳ cách sắp xếp (Timsort của Python cần tới `O(n)` bộ nhớ phụ).

### Cách 3 — Sắp xếp + ghép cặp hai đầu (không cần chọn trung vị)

**Ý tưởng:** theo chứng minh mục 6, đáp án `= Σ (a_{n-1-i} - a_i)` với `i < n/2`. Hai con trỏ `i`, `j` từ hai đầu, cộng `a[j] - a[i]`.

In [12]:
def moves_pairs(nums):
    a = sorted(nums)
    i, j = 0, len(a) - 1
    total = 0
    while i < j:
        total += a[j] - a[i]
        i += 1
        j -= 1
    return total


for nums, expected in examples:
    print(nums, moves_pairs(nums), "kỳ vọng", expected)

[1, 2, 3] 2 kỳ vọng 2
[1, 10, 2, 9] 16 kỳ vọng 16


In [13]:
# Code nộp LeetCode — Cách 3
class Solution3:
    def minMoves2(self, nums: List[int]) -> int:
        nums.sort()
        i, j = 0, len(nums) - 1
        ans = 0
        while i < j:
            ans += nums[j] - nums[i]
            i += 1
            j -= 1
        return ans


print(Solution3().minMoves2([1, 2, 3]), Solution3().minMoves2([1, 10, 2, 9]), Solution3().minMoves2([7]))

2 16 0


**Độ phức tạp Cách 3:** `Time = O(n log n)`, `Space` như sắp xếp. Ưu điểm: **không dùng `abs`** và không phải nghĩ "trung vị chẵn lẻ".

### Cách 4 — Quickselect: tìm trung vị không cần sắp xếp toàn bộ (`O(n)` trung bình)

**Ý tưởng:** ta chỉ cần phần tử thứ `k = n // 2` (đếm từ 0) của mảng đã sắp xếp, không cần cả mảng sắp xếp.

1. Chọn `pivot` ngẫu nhiên, **phân hoạch 3 phần**: `< pivot`, `== pivot`, `> pivot` (3 phần để xử lý nhiều giá trị trùng).
2. Nếu `k < len(less)`: tìm tiếp trong `less`. Nếu `k < len(less) + len(equal)`: đáp án là `pivot`. Ngược lại tìm trong `greater` với `k` trừ đi phần đã bỏ.
3. Có trung vị `k`, tính `Σ|v - k|` như Cách 2.

In [14]:
def quickselect(arr, k, rng):
    """Phần tử nhỏ thứ k (đếm từ 0) của arr, trung bình O(n)."""
    arr = list(arr)
    while True:
        pivot = arr[rng.randrange(len(arr))]
        less = [v for v in arr if v < pivot]
        equal_count = sum(1 for v in arr if v == pivot)
        if k < len(less):
            arr = less
        elif k < len(less) + equal_count:
            return pivot
        else:
            k -= len(less) + equal_count
            arr = [v for v in arr if v > pivot]


def moves_quickselect(nums, seed=0):
    rng = random.Random(seed)
    med = quickselect(nums, len(nums) // 2, rng)
    return sum(abs(v - med) for v in nums)


for nums, expected in examples:
    print(nums, moves_quickselect(nums), "kỳ vọng", expected)
print(quickselect([7, 1, 5, 3, 9], 2, random.Random(1)))   # 5: phần tử nhỏ thứ 2 (đếm từ 0)

[1, 2, 3] 2 kỳ vọng 2
[1, 10, 2, 9] 16 kỳ vọng 16
5


In [15]:
# Code nộp LeetCode — Cách 4
import random as _random

class Solution4:
    def minMoves2(self, nums: List[int]) -> int:
        k = len(nums) // 2
        arr = nums
        while True:
            pivot = arr[_random.randrange(len(arr))]
            less = [v for v in arr if v < pivot]
            eq = sum(1 for v in arr if v == pivot)
            if k < len(less):
                arr = less
            elif k < len(less) + eq:
                med = pivot
                break
            else:
                k -= len(less) + eq
                arr = [v for v in arr if v > pivot]
        return sum(abs(v - med) for v in nums)


print(Solution4().minMoves2([1, 2, 3]), Solution4().minMoves2([1, 10, 2, 9]), Solution4().minMoves2([5, 5, 5]))

2 16 0


**Độ phức tạp Cách 4:** kích thước mảng giảm trung bình theo cấp số nhân ⇒ `Time = O(n)` kỳ vọng (trường hợp xấu nhất `O(n^2)` rất hiếm với pivot ngẫu nhiên), `Space = O(n)` cho các list con.

### Cách 5 — Nhị phân trên giá trị đích (dùng độ dốc của hàm lồi)

**Ý tưởng:** theo mục 3.5, `f(x+1) - f(x) = 2·cnt(≤ x) - n`. Tìm `x` nhỏ nhất có `2·cnt(≤ x) ≥ n` (độ dốc `≥ 0`): đó là **trung vị dưới**. Nhị phân trên đoạn `[min, max]` với mỗi bước đếm `O(n)` ⇒ `O(n log R)`.

In [16]:
def moves_binary_search(nums):
    n = len(nums)
    lo, hi = min(nums), max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        cnt = sum(1 for v in nums if v <= mid)
        if 2 * cnt >= n:             # độ dốc f(mid+1) - f(mid) >= 0: cực tiểu ở mid hoặc bên trái
            hi = mid
        else:
            lo = mid + 1
    return sum(abs(v - lo) for v in nums)


for nums, expected in examples:
    print(nums, moves_binary_search(nums), "kỳ vọng", expected)

[1, 2, 3] 2 kỳ vọng 2
[1, 10, 2, 9] 16 kỳ vọng 16


In [17]:
# Code nộp LeetCode — Cách 5
class Solution5:
    def minMoves2(self, nums: List[int]) -> int:
        n = len(nums)
        lo, hi = min(nums), max(nums)
        while lo < hi:
            mid = (lo + hi) // 2
            if 2 * sum(1 for v in nums if v <= mid) >= n:
                hi = mid
            else:
                lo = mid + 1
        return sum(abs(v - lo) for v in nums)


print(Solution5().minMoves2([1, 2, 3]), Solution5().minMoves2([1, 10, 2, 9]), Solution5().minMoves2([-5, 8]))

2 16 13


**Độ phức tạp Cách 5:** `Time = O(n log R)` với `R = max - min ≤ 2·10^9` (`log R ≈ 31`), `Space = O(1)`. Cách này nổi bật khi **hàm chi phí lồi nhưng không có công thức** trung vị (ví dụ chi phí có trọng số phức tạp).

**Tóm tắt:** Cách 1 để hiểu; Cách 2 là bản nộp chuẩn; Cách 3 không cần `abs`; Cách 4 tránh sắp xếp (`O(n)`); Cách 5 là công cụ tổng quát cho hàm lồi.

## 8. TRACE từng bước

`trace_median` in từng bước của Cách 2 (sắp xếp, chọn trung vị, từng khoảng cách); `trace_pairs` in các cặp của Cách 3; `trace_binary` in từng bước nhị phân của Cách 5.

In [18]:
def trace_median(nums):
    a = sorted(nums)
    n = len(a)
    k = a[n // 2]
    print("sau sắp xếp:", a, "| n =", n, "| chỉ số giữa n//2 =", n // 2, "| trung vị =", k)
    total = 0
    for v in a:
        total += abs(v - k)
        print(f"  |{v} - {k}| = {abs(v - k)}  -> cộng dồn {total}")
    return total


print(trace_median([1, 10, 2, 9]))

sau sắp xếp: [1, 2, 9, 10] | n = 4 | chỉ số giữa n//2 = 2 | trung vị = 9
  |1 - 9| = 8  -> cộng dồn 8
  |2 - 9| = 7  -> cộng dồn 15
  |9 - 9| = 0  -> cộng dồn 15
  |10 - 9| = 1  -> cộng dồn 16
16


In [19]:
def trace_pairs(nums):
    a = sorted(nums)
    i, j = 0, len(a) - 1
    total = 0
    print("sau sắp xếp:", a)
    while i < j:
        total += a[j] - a[i]
        print(f"  i={i} j={j}: a[j]-a[i] = {a[j]} - {a[i]} = {a[j] - a[i]} -> cộng dồn {total}")
        i += 1
        j -= 1
    return total


print(trace_pairs([1, 10, 2, 9, 4]))

sau sắp xếp: [1, 2, 4, 9, 10]
  i=0 j=4: a[j]-a[i] = 10 - 1 = 9 -> cộng dồn 9
  i=1 j=3: a[j]-a[i] = 9 - 2 = 7 -> cộng dồn 16
16


In [20]:
def trace_binary(nums):
    n = len(nums)
    lo, hi = min(nums), max(nums)
    print(f"n={n}, đoạn tìm [{lo}, {hi}]")
    while lo < hi:
        mid = (lo + hi) // 2
        cnt = sum(1 for v in nums if v <= mid)
        decision = "hi = mid" if 2 * cnt >= n else "lo = mid + 1"
        print(f"  lo={lo:>2} hi={hi:>2} mid={mid:>2} cnt(<=mid)={cnt} 2*cnt={2 * cnt:>2} vs n={n} -> {decision}")
        if 2 * cnt >= n:
            hi = mid
        else:
            lo = mid + 1
    print("đích x =", lo)
    return sum(abs(v - lo) for v in nums)


print(trace_binary([1, 10, 2, 9]))

n=4, đoạn tìm [1, 10]
  lo= 1 hi=10 mid= 5 cnt(<=mid)=2 2*cnt= 4 vs n=4 -> hi = mid
  lo= 1 hi= 5 mid= 3 cnt(<=mid)=2 2*cnt= 4 vs n=4 -> hi = mid
  lo= 1 hi= 3 mid= 2 cnt(<=mid)=2 2*cnt= 4 vs n=4 -> hi = mid
  lo= 1 hi= 2 mid= 1 cnt(<=mid)=1 2*cnt= 2 vs n=4 -> lo = mid + 1
đích x = 2
16


Đọc bảng nhị phân với `[1, 10, 2, 9]` (`n = 4`): `mid = 5` có `cnt = 2`, `2·2 = 4 ≥ 4` ⇒ `hi = 5`; `mid = 3`, `cnt = 2` ⇒ `hi = 3`; `mid = 2`, `cnt = 2` ⇒ `hi = 2`; `mid = 1`, `cnt = 1`, `2 < 4` ⇒ `lo = 2`. Đích `2` là trung vị dưới; tổng `1 + 0 + 8 + 7 = 16`. ✓

## 9. Code nộp LeetCode

Bản nộp chính: sắp xếp + trung vị (Cách 2). Ngắn gọn, `O(n log n)` thừa đủ cho `n = 10^5`.

In [21]:
class Solution:
    def minMoves2(self, nums: List[int]) -> int:
        nums.sort()
        k = nums[len(nums) // 2]
        return sum(abs(v - k) for v in nums)


for nums, expected in examples:
    assert Solution().minMoves2(list(nums)) == expected
print("OK các ví dụ đề")

OK các ví dụ đề


## 10. Vì sao thuật toán đúng

**Cận dưới.** Sắp xếp `a_1 ≤ … ≤ a_n`, ghép cặp `(a_i, a_{n+1-i})` với `i = 1..⌊n/2⌋`. Với bất kỳ đích `x`, bất đẳng thức tam giác cho `|a_i - x| + |a_{n+1-i} - x| ≥ a_{n+1-i} - a_i`. Cộng các cặp (phần tử giữa khi `n` lẻ đóng góp `≥ 0`):

`Σ|a_i - x| ≥ L := Σ_{i=1}^{⌊n/2⌋} (a_{n+1-i} - a_i)`.

**Đạt được.** Chọn `x = a_{⌈n/2⌉}` hoặc bất kỳ `x` với `a_{⌊n/2⌋} ≤ x ≤ a_{⌈(n+1)/2⌉}`: với mọi `i ≤ ⌊n/2⌋` có `a_i ≤ a_{⌊n/2⌋} ≤ x ≤ a_{⌈(n+1)/2⌉} ≤ a_{n+1-i}`, nên dấu bằng xảy ra ở **mọi** cặp; phần tử giữa (nếu có) bằng `x` nên đóng góp `0`. Vậy tối thiểu `= L`.

**Đích là số nguyên.** Trung vị `a[n // 2]` là một phần tử của mảng nên là số nguyên, và mỗi lượt `±1` đưa một phần tử `a_i` về `x` đúng `|a_i - x|` lượt (không thể ít hơn, và không cần nhiều hơn). Vậy số lượt tối thiểu bằng `min_x Σ|a_i - x| = L` với `x` nguyên, và không có đích thực không nguyên nào tốt hơn vì `L` đã đạt tại phần tử mảng.

**Các cách khác.** Cách 3 tính đúng `L`. Cách 4 tìm đúng `a[n//2]` rồi tính như Cách 2. Cách 5: hàm `f` lồi nên điểm đầu tiên có độ dốc `f(x+1) - f(x) ≥ 0` là một điểm cực tiểu; và `f(x+1)-f(x) = 2·cnt(≤x) - n` (mỗi phần tử `≤ x` tăng 1 khoảng cách, mỗi phần tử `> x` giảm 1).

Mục 12 còn đối chiếu với **BFS vét cạn trên các trạng thái** (oracle độc lập, không dùng khái niệm trung vị).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi đích | `O(n · R)` | `O(1)` | `R` tới `2·10^9`: chỉ để kiểm chứng |
| 2. Sắp xếp + trung vị | `O(n log n)` | `O(1)`–`O(n)` | **nộp bài** |
| 3. Sắp xếp + ghép cặp | `O(n log n)` | như sắp xếp | không cần `abs` |
| 4. Quickselect | `O(n)` kỳ vọng | `O(n)` | tránh sắp xếp toàn bộ |
| 5. Nhị phân trên giá trị | `O(n log R)` | `O(1)` | tổng quát cho hàm lồi |

Với `n = 10^5`: Cách 2, 3 vài chục mili-giây; Cách 4 nhanh nhất; Cách 5 khoảng `31` lượt quét.

## 12. Bộ test

### 12.1. Oracle độc lập (BFS) + ví dụ đề + edge case

**Oracle** không hề biết tới trung vị: BFS trên các trạng thái mảng (mỗi lượt tăng hoặc giảm một phần tử đi `1`), trong đoạn `[min, max]` ban đầu, đến khi mọi phần tử bằng nhau. Chỉ dùng cho mảng rất nhỏ.

In [22]:
def oracle_bfs(nums):
    lo, hi = min(nums), max(nums)
    start = tuple(nums)
    if len(set(start)) == 1:
        return 0
    seen = {start}
    q = deque([(start, 0)])
    while q:
        state, d = q.popleft()
        for i in range(len(state)):
            for delta in (-1, 1):
                v = state[i] + delta
                if v < lo or v > hi:
                    continue
                nxt = state[:i] + (v,) + state[i + 1:]
                if len(set(nxt)) == 1:
                    return d + 1
                if nxt not in seen:
                    seen.add(nxt)
                    q.append((nxt, d + 1))


solutions = {
    "try_all": moves_try_all,
    "median": moves_median,
    "pairs": moves_pairs,
    "quickselect": moves_quickselect,
    "binary": moves_binary_search,
    "Solution": lambda nums: Solution().minMoves2(list(nums)),
}

edge = [
    ([1, 2, 3], 2),
    ([1, 10, 2, 9], 16),
    ([5], 0),                         # n = 1
    ([7, 7, 7], 0),                   # đã bằng nhau
    ([1, 2], 1),                      # n = 2
    ([-3, 0, 3], 6),                  # số âm
    ([1, 1, 1, 100], 99),             # outlier: median = 1
    ([1, 100, 100, 100], 99),
    ([1, 2, 3, 4], 4),                # n chẵn: đích 2 hoặc 3 đều 4
    ([-5, -5, 5, 5], 20),
    ([0, 0, 0, 0, 7], 7),
    ([3, 2, 1], 2),                   # không sắp xếp
]
for nums, expected in edge:
    for name, fn in solutions.items():
        if name == "try_all" and max(nums) - min(nums) > 1000:
            continue
        assert fn(nums) == expected, (name, nums, fn(nums), expected)
    if max(nums) - min(nums) <= 6 and len(nums) <= 5:
        assert oracle_bfs(nums) == expected, ("bfs", nums)
print("OK", len(edge), "ca, mọi cách đều khớp")

OK 12 ca, mọi cách đều khớp


### 12.2. Phản ví dụ cho các cách làm sai

Ba "lời giải nghe có lý": (1) đích là **trung bình**, (2) lấy phần tử giữa của mảng **chưa sắp xếp**, (3) dùng chỉ số `n // 2 - 1` khi `n` lẻ.

In [23]:
def wrong_mean(nums):
    k = round(sum(nums) / len(nums))
    return sum(abs(v - k) for v in nums)

def wrong_unsorted_middle(nums):
    k = nums[len(nums) // 2]                 # SAI: chưa sắp xếp
    return sum(abs(v - k) for v in nums)

def wrong_index(nums):
    a = sorted(nums)
    k = a[len(a) // 2 - 1]                   # SAI khi n lẻ
    return sum(abs(v - k) for v in a)

tests = {
    "trung bình": (wrong_mean, [1, 2, 100]),
    "giữa chưa sắp xếp": (wrong_unsorted_middle, [9, 1, 2]),
    "chỉ số n//2 - 1": (wrong_index, [1, 2, 100]),
}
for name, (fn, nums) in tests.items():
    print(f"{name:20s} {nums} -> sai: {fn(nums)}, đúng: {moves_median(nums)}")
    assert fn(nums) != moves_median(nums)

trung bình           [1, 2, 100] -> sai: 131, đúng: 99
giữa chưa sắp xếp    [9, 1, 2] -> sai: 9, đúng: 8
chỉ số n//2 - 1      [1, 2, 100] -> sai: 100, đúng: 99


### 12.3. Vét cạn BFS trên mọi mảng nhỏ

Mọi mảng độ dài `1..4` với giá trị `0..3` (tổng `4 + 16 + 64 + 256 = 340` mảng) đối chiếu với oracle BFS.

In [24]:
count = 0
for n in range(1, 5):
    for tup in product(range(4), repeat=n):
        exp = oracle_bfs(tup)
        for name, fn in solutions.items():
            assert fn(list(tup)) == exp, (name, tup, exp)
        count += 1
print("đã đối chiếu", count, "mảng với BFS: khớp hoàn toàn")

đã đối chiếu 340 mảng với BFS: khớp hoàn toàn


### 12.4. Ngẫu nhiên có seed (đúng ràng buộc) và mảng lớn

Sinh mảng `n ≤ 60` với giá trị trong ràng buộc (kể cả sát `±10^9`), nhiều giá trị trùng; đối chiếu Cách 2–5 với nhau và với `try_all` khi khoảng giá trị nhỏ. Rồi `n = 10^5`.

In [25]:
random.seed(462)
fast = [moves_median, moves_pairs, moves_quickselect, moves_binary_search, solutions["Solution"]]
for _ in range(1500):
    n = random.randint(1, 60)
    lo = random.choice([-10 ** 9, -50, 0, 10 ** 9 - 100])
    hi = min(10 ** 9, lo + random.choice([3, 30, 10 ** 4, 2 * 10 ** 9]))
    nums = [random.randint(lo, hi) for _ in range(n)]
    results = {fn(nums) for fn in fast}
    assert len(results) == 1, (nums, results)
    if hi - lo <= 100:
        assert moves_try_all(nums) in results

big = [random.randint(-10 ** 4, 10 ** 4) for _ in range(10 ** 5)]   # giá trị vừa phải để đáp án < 2^31 như đề đảm bảo
for fn in fast:
    t0 = time.perf_counter()
    r = fn(big)
    print(f"{fn.__name__ if hasattr(fn, '__name__') else 'Solution':>22}: {r}  ({time.perf_counter() - t0:.3f}s)")
vals = {fn(big) for fn in fast}
assert len(vals) == 1
print("OK ngẫu nhiên")

          moves_median: 500194404  (0.025s)
           moves_pairs: 500194404  (0.024s)
     moves_quickselect: 500194404  (0.034s)
   moves_binary_search: 500194404  (0.054s)
              <lambda>: 500194404  (0.025s)


OK ngẫu nhiên


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng **trung bình** thay vì trung vị

Trung bình cực tiểu hoá **tổng bình phương** `Σ(a_i - x)^2`, không phải tổng giá trị tuyệt đối. Một giá trị ngoại lai kéo trung bình đi xa.

In [26]:
nums = [1, 2, 100]
print("trung bình ~", round(sum(nums) / len(nums)), "-> tốn", wrong_mean(nums))
print("trung vị   =", sorted(nums)[1], "-> tốn", moves_median(nums))

trung bình ~ 34 -> tốn 131
trung vị   = 2 -> tốn 99


**Đúng:** dùng trung vị (phần tử giữa sau khi sắp xếp).

### Lỗi 2: Lấy phần tử giữa của mảng **chưa sắp xếp**

`nums[n // 2]` chỉ là "phần tử ở giữa theo vị trí", không phải theo giá trị.

In [27]:
nums = [9, 1, 2]
print("sai :", wrong_unsorted_middle(nums), "(chọn", nums[1], ")")
print("đúng:", moves_median(nums), "(chọn", sorted(nums)[1], ")")

sai : 9 (chọn 1 )
đúng: 8 (chọn 2 )


**Đúng:** `nums.sort()` trước khi lấy `nums[n // 2]`.

### Lỗi 3: Nhầm với bài 453 (đích là `min`)

Bài 453 "tăng `n-1` phần tử" có đích `min`; bài này "tăng/giảm **một** phần tử" có đích **trung vị**.

In [28]:
def minmoves_453(nums):
    return sum(nums) - len(nums) * min(nums)

nums = [1, 10, 2, 9]
print("công thức bài 453:", minmoves_453(nums), "| bài 462:", moves_median(nums))

công thức bài 453: 18 | bài 462: 16


**Đúng:** đọc kỹ thao tác. Hai bài giống bề ngoài nhưng thao tác khác nên đáp án khác.

### Lỗi 4: Sai chỉ số giữa khi `n` lẻ (`n // 2 - 1`)

Với `n = 3` thì chỉ số giữa là `1`, không phải `0`. Chọn nhầm phần tử `a[0]` cho đích `1` thay vì `2`.

In [29]:
nums = [1, 2, 100]
print("sai (n//2 - 1):", wrong_index(nums), "| đúng:", moves_median(nums))
print("n chẵn thì cả hai chỉ số giữa đều đúng:", moves_median([1, 2, 3, 4]),
      sum(abs(v - 2) for v in [1, 2, 3, 4]), sum(abs(v - 3) for v in [1, 2, 3, 4]))

sai (n//2 - 1): 100 | đúng: 99
n chẵn thì cả hai chỉ số giữa đều đúng: 4 4 4


**Đúng:** `a[n // 2]` đúng cho cả `n` chẵn lẫn lẻ.

### Lỗi 5: Thử mọi đích rồi bị "Time Limit Exceeded"

Cách 1 là `O(n · R)`. Đo thử:

In [30]:
nums = [random.randint(0, 3000) for _ in range(2000)]
t0 = time.perf_counter(); r1 = moves_try_all(nums); t1 = time.perf_counter() - t0
t0 = time.perf_counter(); r2 = moves_median(nums); t2 = time.perf_counter() - t0
print(r1 == r2, f"thử mọi đích: {t1:.3f}s | trung vị: {t2:.5f}s")

True thử mọi đích: 0.419s | trung vị: 0.00039s


Với `R = 2·10^9` cách thử mọi đích hoàn toàn không chạy nổi. **Đúng:** biết đích tối ưu là trung vị.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **"đưa mọi phần tử về một giá trị, mỗi lượt ±1"** ⇒ tổng `|a_i - x|` ⇒ **trung vị**.
- Thấy **"tổng bình phương"** ⇒ trung bình; **"khoảng cách lớn nhất"** ⇒ trung điểm `(min + max) / 2`; **"tổng tuyệt đối"** ⇒ trung vị. Ba "trung tâm" của ba chuẩn khác nhau.
- Mẹo tính nhanh: sắp xếp rồi cộng `a[n-1-i] - a[i]` cho `i < n/2`: không phải nghĩ trung vị chẵn lẻ.
- Cần `O(n)`: `quickselect` hay `statistics.median_low` (nhưng `statistics.median_low` vẫn sắp xếp bên trong).
- Cùng họ: 296 (Best Meeting Point, 2D Manhattan), 2448 (Minimum Cost to Make Array Equal, trung vị có trọng số), 453, 1131, 2033.

## 15. Biến thể của bài

1. **Trung vị có trọng số** (mỗi phần tử có chi phí `cost[i]` mỗi bước): LeetCode 2448.
2. **Điểm hẹn 2D với khoảng cách Manhattan** (LeetCode 296): tách hai trục, lấy trung vị mỗi trục.
3. **Tổng bình phương** (L2): đích là **trung bình**, tổng nhỏ nhất `Σ(a_i - mean)^2`.
4. **Hỏi đích tối ưu** thay vì số lượt: trả về một trung vị.

Dưới đây là biến thể 1, 2, 3 (có test so với vét cạn).

In [31]:
def weighted_min_cost(nums, cost):
    """Mỗi bước trên phần tử i tốn cost[i]. Đích = trung vị có trọng số."""
    pairs = sorted(zip(nums, cost))
    total = sum(cost)
    acc = 0
    for v, c in pairs:
        acc += c
        if 2 * acc >= total:
            target = v
            break
    return sum(abs(v - target) * c for v, c in pairs)


def meeting_point_2d(points):
    """Mỗi điểm (r, c) đi 4 hướng; tổng quãng đường Manhattan nhỏ nhất tới một điểm."""
    rs = sorted(p[0] for p in points)
    cs = sorted(p[1] for p in points)
    mr, mc = rs[len(rs) // 2], cs[len(cs) // 2]
    return sum(abs(r - mr) for r in rs) + sum(abs(c - mc) for c in cs)


def min_sum_squares(nums):
    """Tổng bình phương nhỏ nhất với đích nguyên: thử floor/ceil của trung bình."""
    m = sum(nums) // len(nums)
    return min(sum((v - x) ** 2 for v in nums) for x in (m, m + 1))


random.seed(15)
for _ in range(300):
    n = random.randint(1, 7)
    nums = [random.randint(-6, 6) for _ in range(n)]
    cost = [random.randint(1, 5) for _ in range(n)]
    exp = min(sum(abs(v - x) * c for v, c in zip(nums, cost)) for x in range(-6, 7))
    assert weighted_min_cost(nums, cost) == exp, (nums, cost)
    assert min_sum_squares(nums) == min(sum((v - x) ** 2 for v in nums) for x in range(-6, 7))
    pts = [(random.randint(0, 5), random.randint(0, 5)) for _ in range(n)]
    exp2 = min(sum(abs(r - a) + abs(c - b) for r, c in pts) for a in range(6) for b in range(6))
    assert meeting_point_2d(pts) == exp2, pts
print(weighted_min_cost([1, 3, 5], [2, 3, 1]), meeting_point_2d([(0, 0), (0, 4), (2, 2)]), min_sum_squares([1, 2, 100]))
print("OK biến thể")

6 6 6469
OK biến thể


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi đích | tính `f(x)` cho mọi `x` | `O(n·R)` | `O(1)` | chỉ để kiểm chứng |
| 2. Sắp xếp + trung vị | đích = `a[n//2]` | `O(n log n)` | `O(1)`–`O(n)` | **nộp bài**, ngắn nhất |
| 3. Sắp xếp + ghép cặp | cộng `a[j]-a[i]` | `O(n log n)` | như sắp xếp | không cần `abs`, tránh nghĩ chẵn lẻ |
| 4. Quickselect | tìm phần tử thứ `n//2` | `O(n)` kỳ vọng | `O(n)` | `n` rất lớn, tránh sắp xếp |
| 5. Nhị phân trên giá trị | dùng độ dốc `2·cnt - n` | `O(n log R)` | `O(1)` | hàm chi phí lồi tổng quát |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Solution().minMoves2([4, 4, 4])`, `Solution().minMoves2([1, 5])`, `Solution().minMoves2([3, 9, 1])`, `moves_pairs([2, 8, 4, 6])`. Điền `ans1` là list bốn số (hiện `None`).

**Bài 2 (tính tay).** Với `nums = [7, 1, 5, 3, 11]`: sắp xếp, tìm trung vị, tính từng `|v - trung vị|` và tổng. Điền `ans2_median`, `ans2_total`.

**Bài 3 (điền TODO).** Hoàn thành `moves_todo(nums)` theo Cách 3 (ghép cặp hai đầu).

In [32]:
ans1 = None
ans2_median = None
ans2_total = None

def moves_todo(nums):
    # TODO 1: a = sorted(nums)
    # TODO 2: i, j = 0, len(a) - 1; total = 0
    # TODO 3: while i < j: total += a[j] - a[i]; i += 1; j -= 1
    # TODO 4: return total
    pass

# KHÔNG gọi moves_todo ở đây để Run All không lỗi.

**Bài 4 (tự viết lại).** Viết `moves_binary(nums)` theo Cách 5 từ đầu: nhị phân trên `[min, max]`, điều kiện `2 * (số phần tử <= mid) >= n`, rồi tính tổng khoảng cách.

**Bài 5 (biến thể / mini challenge).** (a) Viết `moves_with_target_limit(nums, lo, hi)`: đích bắt buộc nằm trong `[lo, hi]` (kẹp trung vị vào đoạn đó). (b) Viết `moves_k_equal(nums, k)`: số lượt ít nhất để có **ít nhất `k` phần tử** bằng nhau (không cần cả mảng). Gợi ý: sắp xếp, trượt cửa sổ `k` phần tử liên tiếp, mỗi cửa sổ tính tổng hiệu cặp.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[4,4,4]` đã bằng nhau; `[1,5]` chênh `4`; `[3,9,1]` sắp xếp `[1,3,9]`, trung vị `3`; `[2,8,4,6]` sắp xếp `[2,4,6,8]` ghép `(2,8)`, `(4,6)`.
- **Mức 2 (vừa):** Bài 2: sắp xếp `[1,3,5,7,11]`, chỉ số giữa `5 // 2 = 2` ⇒ trung vị `5`. Bài 3: hai con trỏ, dừng khi `i >= j`.
- **Mức 3 (sâu):** Bài 5(a): `target = min(max(median, lo), hi)` do hàm lồi, điểm gần trung vị nhất trong đoạn là điểm tối ưu. Bài 5(b): trong mỗi cửa sổ `k` phần tử đã sắp xếp, chi phí là tổng hiệu cặp đầu-cuối; dùng vòng lặp `O(n·k)` hoặc tiền tố `O(n)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
ans1 = [0, 4, 8, 8]
assert ans1 == [Solution().minMoves2([4, 4, 4]), Solution().minMoves2([1, 5]),
                Solution().minMoves2([3, 9, 1]), moves_pairs([2, 8, 4, 6])]

# Bài 2: [1,3,5,7,11], trung vị 5, |.|: 4+2+0+2+6 = 14
ans2_median, ans2_total = 5, 14
s2 = sorted([7, 1, 5, 3, 11])
assert ans2_median == s2[len(s2) // 2]
assert ans2_total == sum(abs(v - ans2_median) for v in s2) == Solution().minMoves2([7, 1, 5, 3, 11])

# Bài 3
def moves_todo(nums):
    a = sorted(nums)
    i, j = 0, len(a) - 1
    total = 0
    while i < j:
        total += a[j] - a[i]
        i += 1
        j -= 1
    return total

# Bài 4
def moves_binary(nums):
    n = len(nums)
    lo, hi = min(nums), max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if 2 * sum(1 for v in nums if v <= mid) >= n:
            hi = mid
        else:
            lo = mid + 1
    return sum(abs(v - lo) for v in nums)

random.seed(19)
for _ in range(300):
    nums = [random.randint(-20, 20) for _ in range(random.randint(1, 9))]
    assert moves_todo(nums) == moves_binary(nums) == Solution().minMoves2(list(nums))
print("OK bài 1-4")

OK bài 1-4


In [34]:
# Bài 5(a): đích bị giới hạn trong [lo, hi]
def moves_with_target_limit(nums, lo, hi):
    a = sorted(nums)
    med = a[len(a) // 2]
    target = min(max(med, lo), hi)
    return sum(abs(v - target) for v in a)

random.seed(51)
for _ in range(300):
    nums = [random.randint(-10, 10) for _ in range(random.randint(1, 8))]
    lo = random.randint(-10, 10)
    hi = random.randint(lo, 10)
    exp = min(sum(abs(v - x) for v in nums) for x in range(lo, hi + 1))
    assert moves_with_target_limit(nums, lo, hi) == exp, (nums, lo, hi)

# Bài 5(b): ít nhất k phần tử bằng nhau
def moves_k_equal(nums, k):
    a = sorted(nums)
    best = None
    for s in range(len(a) - k + 1):
        w = a[s:s + k]
        cost = sum(w[len(w) - 1 - t] - w[t] for t in range(len(w) // 2))
        if best is None or cost < best:
            best = cost
    return best

for _ in range(300):
    nums = [random.randint(-8, 8) for _ in range(random.randint(1, 7))]
    k = random.randint(1, len(nums))
    exp = min(sum(abs(v - x) for v in sorted(nums, key=lambda y: abs(y - x))[:k]) for x in range(-8, 9))
    assert moves_k_equal(nums, k) == exp, (nums, k)
assert moves_k_equal([1, 2, 3, 100], 3) == 2
print("OK bài 5")

OK bài 5


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tổng khoảng cách nhỏ nhất ⇒ trung vị)

```python
class Solution:
    def minMoves2(self, nums: List[int]) -> int:
        nums.sort()
        k = nums[len(nums) // 2]           # trung vị
        return sum(abs(v - k) for v in nums)
```

Biến thể không cần trung vị: `sum(nums[~i] - nums[i] for i in range(len(nums) // 2))` sau khi sắp xếp (`~i` là `-i-1`).

**Checklist**

- [ ] Thao tác là "tăng/giảm **một** phần tử đi 1" ⇒ đích là **trung vị** (không phải `min`, không phải trung bình)?
- [ ] Đã **sắp xếp** trước khi lấy phần tử giữa?
- [ ] Chỉ số giữa `n // 2` đúng cho cả `n` chẵn và `n` lẻ?
- [ ] Đã thử `[1,2,3]` (2), `[1,10,2,9]` (16), `[5]` (0), `[1,2,100]` (99), `[1,2,3,4]` (4)?
- [ ] Hiểu vì sao hai đầu ghép cặp cho cận dưới `a_max - a_min`?
- [ ] Skeleton đúng `class Solution` / `def minMoves2(self, nums: List[int]) -> int`?